## 依赖导入

本例演示回归分析的常用计算：一元/多元线性回归（公式接口与矩阵接口 OLS）、多项式回归、逐步回归、Logistic 回归（线性化最小二乘、GLM 二项族、极大似然 logit）与 Probit 回归，以及参数显著性的 F 检验、t 检验和残差野值（离群点）诊断。绘图用 matplotlib，数值用 numpy/pandas，回归与广义线性模型用 statsmodels，分位数取自 scipy.stats 的 F、正态、t 分布。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import f, norm, t

## 全局绘图配置

设置全局 `plt.rc`：字号 15/16、启用 TeX 渲染数学公式，保证后续各图的公式与散点标签正常显示。

In [ ]:
plt.rc("font", size=15)
plt.rc("text", usetex=True)
plt.rc("font", size=16)

## 8.1  一元线性回归与野值诊断

定义函数 check(d)：取数据第 1、2 行为自变量 x0 与因变量 y0，用公式接口 `sm.formula.ols('y~x', d)` 拟合线性回归模型；打印回归摘要、`outlier_test()` 的野值（离群点）检验以及残差方差 `mse_resid`。再用 `get_prediction` 取 `summary_frame(alpha=0.05)` 中第 5 列起的两列作为置信上下限 low/upp，算出置信半径 $r=(\text{upp}-\text{low})/2$，并以 `errorbar` 画出各观测点残差及其置信半径。随后依次删除第 9 列（`axis=1`，索引 8）与第 5 列（索引 4）后重复检验，演示逐步剔除野值的过程（源程序 ex10_1.py）。

In [ ]:
def check(d):
    x0 = d[0]
    y0 = d[1]
    d = {"x": x0, "y": y0}
    re = sm.formula.ols("y~x", d).fit()
    print(re.summary())
    print(re.outlier_test())
    print("残差的方差", re.mse_resid)
    pre = re.get_prediction(d)
    df = pre.summary_frame(alpha=0.05)
    dfv = df.values
    low, upp = dfv[:, 4:].T
    r = (upp - low) / 2
    num = np.arange(1, len(x0) + 1)
    _fig, ax = plt.subplots()
    ax.errorbar(num, re.resid, r, fmt="o")
    plt.show()


a = np.loadtxt("data/10_1.txt")
check(a)
a2 = np.delete(a, 8, axis=1)
check(a2)
a3 = np.delete(a2, 4, axis=1)
check(a3)

## 8.1  一元线性回归与野值诊断（单步对照）

将上述诊断过程内联展开（不经 check 函数、不删列）：仅对完整数据拟合 `y~x`，打印回归摘要、野值检验与残差方差，并按预测置信上下限画残差-观测序号的误差棒图，作为 ex10_1.py 的单步对照（源程序 ex10_1_2.py）。

In [ ]:
d = np.loadtxt("data/10_1.txt")
x0 = d[0]
y0 = d[1]
d = {"x": x0, "y": y0}
re = sm.formula.ols("y~x", d).fit()
print(re.summary())
print(re.outlier_test())
print("残差的方差", re.mse_resid)
pre = re.get_prediction(d)
df = pre.summary_frame(alpha=0.05)
dfv = df.values
low, upp = dfv[:, 4:].T
r = (upp - low) / 2
num = np.arange(1, len(x0) + 1)
fig, ax = plt.subplots()
ax.errorbar(num, re.resid, r, fmt="o")
plt.show()

## 8.2  二元线性回归（公式接口）

读取 `data/10_2.txt`（三行：$x_1$、$x_2$、$y$），在同一子图分别用星号、圆圈画出 $x_1$、$x_2$ 对 $y$ 的散点并加图例；再用公式接口 `sm.formula.ols('y~x1+x2', d)` 拟合二元线性回归，打印回归摘要与残差方差 `mse_resid`，并对点 $(9,10)$、$(10,9)$ 给出预测值 yh（源程序 ex10_2_1.py）。

In [ ]:
a = np.loadtxt("data/10_2.txt")
fig, ax = plt.subplots()
ax.plot(a[0], a[2], "*", label="$x_1$")
ax.plot(a[1], a[2], "o", label="$x_2$")
ax.legend(loc="upper left")
d = {"x1": a[0], "x2": a[1], "y": a[2]}
re = sm.formula.ols("y~x1+x2", d).fit()
print(re.summary())
yh = re.predict({"x1": [9, 10], "x2": [10, 9]})
print("残差的方差:", re.mse_resid)
print("预测值：", yh)
plt.show()

## 8.2  二元线性回归（矩阵接口 OLS）

同一数据改用矩阵接口：用 `sm.add_constant(a[:2].T)` 为 $x_1,x_2$ 添加常数列后调用 `sm.OLS` 拟合，再对含常数列的设计矩阵 `[[1,9,10],[1,10,9]]` 预测 $(9,10)$、$(10,9)$，打印摘要、残差方差与预测值，作为与公式接口等价的实现（源程序 ex10_2_2.py）。

In [ ]:
a = np.loadtxt("data/10_2.txt")
fig, ax = plt.subplots()
ax.plot(a[0], a[2], "*", label="$x_1$")
ax.plot(a[1], a[2], "o", label="$x_2$")
ax.legend(loc="upper left")
X = sm.add_constant(a[:2].T)
re = sm.OLS(a[2], X).fit()
print(re.summary())
yh = re.predict(np.array([[1, 9, 10], [1, 10, 9]]))
print("残差的方差:", re.mse_resid)
print("预测值：", yh)
plt.show()

## 8.3  多项式（二次）回归

以 17~29 的奇数为自变量 $x$（`np.arange(17, 30, 2)`），读取 `data/10_3.txt` 的两列观测 $y_1,y_2$，在同一子图画两组散点；再把 $x$ 与两组 $y$ 分别复制拼接（`np.hstack`、`a.flatten()`）构造样本，用 `smf.ols('y~x+I(x**2)', d)` 拟合二次多项式回归，打印摘要与残差方差（源程序 ex10_3.py）。

In [ ]:
x = np.arange(17, 30, 2)
a = np.loadtxt("data/10_3.txt")
fig, ax = plt.subplots()
ax.plot(x, a[0], "*", label="$y_1$")
ax.plot(x, a[1], "o", label="$y_2$")
x = np.hstack([x, x])
d = {"y": a.flatten(), "x": x}
re = smf.ols("y~x+I(x**2)", d).fit()
print(re.summary())
print("残差的方差：", re.mse_resid)
ax.legend()
plt.show()

## 8.4  多项式模型结构比较

读取 `data/10_4.txt`（三行 $x_1,x_2,y$）画散点，分别用线性模型 `y~x1+x2`、纯二次 `y~x1+x2+I(x1**2)+I(x2**2)`、交叉 `y~x1*x2`、完全二次 `y~x1*x2+I(x1**2)+I(x2**2)` 四种结构拟合，逐一打印各自的残差方差进行对比；再对纯二次模型在 $(x_1,x_2)=(170,160)$ 处预测并打印其完整摘要，据此选择最优模型结构（源程序 ex10_4.py）。

In [ ]:
a = np.loadtxt("data/10_4.txt")
x1 = a[0]
x2 = a[1]
y = a[2]
fig, ax = plt.subplots()
ax.plot(x1, y, "*", label="$x_1$")
ax.plot(x2, y, "o", label="$x_2$")
d = {"y": y, "x1": x1, "x2": x2}
re1 = smf.ols("y~x1+x2", d).fit()
print("线性回归的残差方差：", re1.mse_resid)
re2 = smf.ols("y~x1+x2+I(x1**2)+I(x2**2)", d).fit()
print("纯二次的残差方差：", re2.mse_resid)
re3 = smf.ols("y~x1*x2", d).fit()
print("交叉的残差方差：", re3.mse_resid)
re4 = smf.ols("y~x1*x2+I(x1**2)+I(x2**2)", d).fit()
print("完全二次的残差方差：", re4.mse_resid)
print("预测值：", re2.predict({"x1": 170, "x2": 160}))
print(re2.summary())
ax.legend()
plt.show()

## 8.5  多元回归：矩阵最小二乘、F/t 检验与完全二次模型

读取 `data/10_5.xlsx`，拼接观测向量 Y 与设计矩阵（`np.hstack` 加常数列得 XX，样本容量 25、变量数 $m=3$）。用 `np.linalg.pinv(XX) @ Y` 以最小二乘求回归参数；令 $y_b$ 为 Y 均值、$y_h=XX@cs$ 为估计值，计算残差平方和 $q=\sum(y_h-y)^2$ 与回归平方和 $u=\sum(y_h-\bar y)^2$，构造 $F=u/m/(q/(n-m-1))$ 并与 $F$ 分布上 $\alpha$ 分位数比较；再由 $(XX^TXX)^{-1}$ 的对角元构造 $t$ 统计量并与 $t$ 分布上 $\alpha/2$ 分位数比较。删除 $x_1$ 列重新拟合得 $x_2,x_3$ 子模型的参数；最后用公式接口拟合含交互与平方项的完全二次模型并输出系数（源程序 ex10_5.py）。

In [ ]:
a = pd.read_excel("data/10_5.xlsx", header=None)
b = a.values
Y = np.hstack([b[:, 1], b[:-1, 6]])
X = np.vstack([b[:, 2:5], b[:-1, 7:]])
XX = np.hstack([np.ones((25, 1)), X])
cs = np.linalg.pinv(XX) @ Y
print("拟合的参数为：", np.round(cs, 4))
yb = Y.mean()
yh = XX @ cs
q = sum((yh - Y) ** 2)
u = sum((yh - yb) ** 2)
m = 3
n = len(Y)
F = u / m / (q / (n - m - 1))
print("F=", round(F, 4))
fw = f.ppf(0.95, m, n - m - 1)
print("F 分布的上 alpha 分位数：", round(fw, 4))
c = np.diag(np.linalg.inv(XX.T @ XX))
ts = cs / np.sqrt(c) / np.sqrt(q / (n - m - 1))
tw = t.ppf(0.975, n - m - 1)
print("t 统计量值为：", np.round(ts, 4))
print("t 分布的上 alpha/2 分位数：", round(tw, 4))
XD = np.delete(XX, 1, axis=1)
cs2 = np.linalg.pinv(XD) @ Y
print("x2,x3 模型的参数值：", np.round(cs2, 4))
d = {"y": Y, "x1": X[:, 0], "x2": X[:, 1], "x3": X[:, 2]}
md = sm.formula.ols(
    "y~x1*x2+x1*x3+x2*x3+I(x1**2)+\
                    I(x2**2)+I(x3**2)",
    d,
).fit()
print("完全二次项的系数为：", md.params)

## 8.6  逐步回归（按 t 值后向剔除变量）

读取 `data/10_6.txt`（$x_1..x_4$ 与 $y$ 共 5 列），先拟合全模型 `y~x1+x2+x3+x4` 打印各自变量的 $t$ 值，据此删除 $|t|$ 最小者得 `y~x1+x2+x4` 并再打印 $t$ 值，继续删除不显著者得 `y~x1+x2`，最后打印该模型的完整摘要与残差方差，演示后向剔除（backward elimination）的变量选择过程（源程序 ex10_6.py）。

In [ ]:
a = np.loadtxt("data/10_6.txt")
d = {"x1": a[:, 0], "x2": a[:, 1], "x3": a[:, 2], "x4": a[:, 3], "y": a[:, 4]}
md1 = sm.formula.ols("y~x1+x2+x3+x4", d).fit()
print(md1.tvalues)
md2 = sm.formula.ols("y~x1+x2+x4", d).fit()
print(md2.tvalues)
md3 = sm.formula.ols("y~x1+x2", d).fit()
print(md3.summary())
print("残差方差：", md3.mse_resid)

## 8.7  Logistic 回归：线性化最小二乘

读取 `data/10_7_1.txt`（第 0 列 $x$、第 1 列总数 $n$、第 2 列成功数），计算比例 $p_i=c/n$，取 logit 变换 $y_i=\ln(p_i/(1-p_i))$ 作响应，用 `sm.add_constant(x)` 构造设计阵后 `sm.OLS` 线性拟合（构建并拟合模型），打印模型全部结果；再由拟合系数在 $x=9$ 处反变换得所求比例 $p_0=1/(1+e^{-\hat y})$ 并输出，最后用 `np.savetxt` 把回归系数保存到 `data/10_7_2.txt` 供后续调用（源程序 ex10_7_1.py）。

In [ ]:
a = np.loadtxt("data/10_7_1.txt")
x = a[:, 0]
pi = a[:, 2] / a[:, 1]
X = sm.add_constant(x)
yi = np.log(pi / (1 - pi))
md = sm.OLS(yi, X).fit()
print(md.summary())
p0 = 1 / (1 + np.exp(-md.predict([1, 9])))
b = md.params
print(f"所求比例 p0={p0:.4f}")
np.savetxt("data/10_7_2.txt", b)

## 8.7  Logistic 回归：GLM 二项族

同一数据改用广义线性模型：用 `np.vstack` 把每个观测组织成 (成功次数, 失败次数) 两列作为响应 y（第 1 列成功、第 2 列失败），调用 `sm.formula.glm('y~x', d, family=sm.families.Binomial())` 拟合二项 Logistic 回归并打印摘要，作为线性化最小二乘方法的对照（源程序 ex10_7_2.py）。

In [ ]:
a = np.loadtxt("data/10_7_1.txt")
x = a[:, 0]
y = np.vstack([a[:, 2], a[:, 1] - a[:, 2]]).T
d = {"x": x, "y": y}
md = sm.formula.glm("y~x", d, family=sm.families.Binomial()).fit()
print(md.summary())

## 8.8  二元逻辑回归（logit 极大似然）

读取 `data/10_8.txt`，把两部分样本分别堆叠成自变量矩阵 x（列 $x_1,x_2,x_3$）与响应向量 y；用 `sm.formula.logit('y~x1+x2+x3', d)` 做极大似然拟合并打印摘要；再删去 $x_2$ 拟合 `y~x1+x3` 打印摘要，通过两次结果对比判断变量显著性（源程序 ex10_8_1.py）。

In [ ]:
a = np.loadtxt("data/10_8.txt")
x = np.vstack([a[:, 1:4], a[:, 6:-1]])
y = np.hstack([a[:, 4], a[:, 9]])
d = {"x1": x[:, 0], "x2": x[:, 1], "x3": x[:, 2], "y": y}
md = sm.formula.logit("y~x1+x2+x3", d).fit()
print(md.summary())
md2 = sm.formula.logit("y~x1+x3", d).fit()
print(md2.summary())

## 8.8  二元逻辑回归（GLM 实现）

同一数据改用 `sm.formula.glm(..., family=sm.families.Binomial())` 分别拟合含/不含 $x_2$ 的二项 Logistic 模型并打印摘要，与 logit 极大似然接口相互印证（源程序 ex10_8_2.py）。

In [ ]:
a = np.loadtxt("data/10_8.txt")
x = np.vstack([a[:, 1:4], a[:, 6:-1]])
y = np.hstack([a[:, 4], a[:, 9]])
d = {"x1": x[:, 0], "x2": x[:, 1], "x3": x[:, 2], "y": y}
md1 = sm.formula.glm("y~x1+x2+x3", d, family=sm.families.Binomial()).fit()
print(md1.summary())
md2 = sm.formula.glm("y~x1+x3", d, family=sm.families.Binomial()).fit()
print(md2.summary())

## 8.9  Probit 回归

读取 `data/10_7_1.txt`，对比例 $p_i$ 取标准正态分位数 $y_i=\Phi^{-1}(p_i)$ 作响应，用 `sm.add_constant(x)` 与 `sm.OLS` 对 $x$ 线性拟合（Probit 的线性化解法，构建并拟合模型），打印模型全部结果；再用正态 CDF 求 $x=9$ 处的所求比例 $p_0=\Phi(\hat y)$（源程序 ex10_9.py）。

In [ ]:
a = np.loadtxt("data/10_7_1.txt")
x = a[:, 0]
pi = a[:, 2] / a[:, 1]
yi = norm.ppf(pi)
X = sm.add_constant(x)
md = sm.OLS(yi, X).fit()
print(md.summary())
p0 = norm.cdf(md.predict([1, 9]))
print(f"所求比例 p0={p0:.4f}")

## 8.10  Logistic 系数的优势比

读取 ex10_7_1.py 保存的系数 `data/10_7_2.txt`，计算 $x=9$ 时的优势 $\text{odds}=e^{b\cdot(1,9)}$，以及 $x=9$ 相对 $x=8$ 的优势比 odds9vs8（两者之比），打印这两个值（源程序 ex10_10.py）。

In [ ]:
b = np.loadtxt("data/10_7_2.txt")
odds9 = np.exp(b @ [1, 9])
odds9vs8 = np.exp(b @ [1, 9]) / np.exp(b @ [1, 8])
print(f"odds9={odds9:.4f},odds9vs8={odds9vs8:.4f}")

## 8.11  Logistic 回归（BFGS 优化）

读取 `data/10_11.txt`（三个自变量与响应 $y$），用 `sm.formula.logit('y~x1+x2+x3', d)` 建模；因默认牛顿法不收敛出错，改用 `method='bfgs'` 拟合并打印摘要；最后对给定两组新样本求预测概率（源程序 ex10_11.py）。

In [ ]:
a = np.loadtxt("data/10_11.txt")
d = {"x1": a[:, 0], "x2": a[:, 1], "x3": a[:, 2], "y": a[:, -1]}
md = sm.formula.logit("y~x1+x2+x3", d)
md = md.fit(method="bfgs")
print(md.summary())
print(
    md.predict({"x1": [49.2, 40.6], "x2": [-17.2, 26.4], "x3": [0.3, 1.8]})
)